In [30]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import pytz
from tqdm import tqdm
from datetime import timedelta
from dateutil.relativedelta import relativedelta

In [31]:
sales = pd.read_csv('pedidos-1743765883268.csv', sep=';', encoding='latin1')

In [32]:
df = sales[['account_id', 'sales_channel_id']].value_counts()
df = pd.DataFrame(df).reset_index()

In [33]:
# EPOCHS = 1000
# BATCH_SIZE = 32
# VALID_SPLIT = 0.1

In [34]:
SAO_PAULO_TZ = pytz.timezone('America/Sao_Paulo')
LOOKBACK = 1
# SEASONAL_PERIODS = (24, 24*7)
# COVERAGE = 0.33
END_DATE = pd.to_datetime(pd.to_datetime(sales['created_date'].max()).strftime("%Y-%m-%d %H:00:00"))
# START_DATE = END_DATE - timedelta(hours=END_DATE.hour)
START_DATE = END_DATE - timedelta(hours=23)
# TEST_SIZE = int((END_DATE - START_DATE).seconds / 60**2 + 1)

In [35]:
# OOT_DATE = START_DATE + timedelta(hours=23)

In [36]:
# OOT_PERIODS = int((OOT_DATE - END_DATE).seconds / 60**2 + 1)

In [37]:
# oot_dates = pd.DatetimeIndex([END_DATE+timedelta(hours=h) for h in range(1, OOT_PERIODS)], freq='h')
# df_oot = pd.DataFrame(index=oot_dates)

In [38]:
# df.drop('count', axis=1, inplace=True)
# for account_id in df['account_id'].unique():
#     df = pd.concat([df, pd.DataFrame({'account_id': [account_id], 'sales_channel_id': ['ALL']})], ignore_index=True)

In [39]:
# df

In [40]:
id_pairs = list(zip(df['account_id'], df['sales_channel_id']))
# id_pairs = list(zip(df_filtered['account_id'], df_filtered['sales_channel_id']))

In [41]:
# weights_chan = {}
# for account_id, sales_channel_id in tqdm(id_pairs):
#     # if sales_channel_id == 'ALL':
#     #     cond = (sales['account_id'] == account_id) & \
#     #            (sales['status'].notna())
#     # else:
#     #     cond = (sales['account_id'] == account_id) & \
#     #            (sales['sales_channel_id'] == sales_channel_id) & \
#     #            (sales['status'].notna())
#     cond = (sales['account_id'] == account_id) & \
#            (sales['sales_channel_id'] == sales_channel_id) & \
#            (sales['status'].notna())

#     df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
#     df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
#     df_client = df_client.sort_values('created_date').reset_index(drop=True)
#     df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

#     df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
#     df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

#     end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
#     start_date = end_date - relativedelta(months=LOOKBACK)
#     date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
#     df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
    
#     weights_chan[account_id] = {} if account_id not in weights_chan else weights_chan[account_id]
    
#     df = df_client_mod['n_orders'].fillna(0)
#     df = df.loc[df.index < START_DATE].copy()
    
#     weights_chan[account_id][sales_channel_id] = df.median()

In [42]:
# weights_df = pd.DataFrame(weights_chan)

# weights_df

In [43]:
# weights_df = weights_df.stack().reset_index().rename(columns={'level_0': 'sales_channel_id', 'level_1': 'account_id', 0: 'weights'}).sort_values(by=['account_id', 'sales_channel_id'])

# weights_df

In [44]:
# weights_chan_df = weights_df.groupby('account_id')['weights'].apply(lambda x: x / x.sum()).fillna(0).reset_index().rename(columns={0: 'weights'}).drop('level_1', axis=1)

# weights_chan_df = pd.concat([weights_df['sales_channel_id'].reset_index(drop=True), weights_chan_df], axis=1)

# weights_chan_df

In [45]:
# weights_acc_df = weights_df.groupby('account_id')['weights'].sum() / weights_df.groupby('account_id')['weights'].sum().sum()

# weights_acc_df

In [46]:
# weights_acc_df.sort_values(ascending=False)

In [47]:
weights_chan_df = joblib.load('weights_chan_df.pkl')
weights_acc_df = joblib.load('weights_acc_df.pkl')

In [48]:
dbname = 'railway'
username = 'sinatra'
pwd = '781B3XjpeuqE'
hostname = 'monorail.proxy.rlwy.net'
port = 25096

connection = psycopg2.connect(database=dbname, user=username, password=pwd, host=hostname, port=port)
cursor = connection.cursor()

In [49]:
cursor.execute("select distinct account_id, channel from public.forecast where model='TBATS';")
res_tbats = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='GradientBoosting';")
res_gb = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='LSTM';")
res_lstm = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='Ensemble';")
res_ens = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)';")
res_chronos1 = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='Chronos';")
res_chronos2 = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='MEDIAN_MAD_60';")
res_median = cursor.fetchall()

In [50]:
id_pairs = (
    set(res_tbats)
    .intersection(set(res_gb))
    .intersection(set(res_lstm))
    .intersection(set(res_ens))
    .intersection(set(res_chronos1))
    .intersection(set(res_chronos2))
    .intersection(set(res_median))
)

# id_pairs = set(res_tbats).intersection(set(res_gb))

In [51]:
models = ['amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)', 'MEDIAN_MAD_60']

In [52]:
df_forecast_metrics1_norm, df_forecast_metrics2_norm = {}, {}
scores1, scores2 = {}, {}
for model in models:
    print('\n\n')
    print(model)
    df_forecast_metrics1_norm[model], df_forecast_metrics2_norm[model] = {}, {}
    for account_id, sales_channel_id in id_pairs:
        print(account_id, sales_channel_id)
        
        df_forecast_metrics1_norm[model][account_id] = {} if account_id not in df_forecast_metrics1_norm[model] else df_forecast_metrics1_norm[model][account_id]
        df_forecast_metrics2_norm[model][account_id] = {} if account_id not in df_forecast_metrics2_norm[model] else df_forecast_metrics2_norm[model][account_id]
        
        if sales_channel_id == 'ALL':
            cond = (sales['account_id'] == account_id) & \
                   (sales['status'].notna())
        else:
            sales_channel_id = int(sales_channel_id)
            cond = (sales['account_id'] == account_id) & \
                   (sales['sales_channel_id'] == sales_channel_id) & \
                   (sales['status'].notna())

        df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
        df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
        df_client = df_client.sort_values('created_date').reset_index(drop=True)
        df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

        df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
        df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')
        
        end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
        start_date = end_date - relativedelta(months=LOOKBACK)
        date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
        df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
        
        cursor.execute(f"select start, sales_high, sales_low, sales_mean, orders_high, orders_low, orders_mean from public.forecast where model='{model}' and account_id='{account_id}' and channel='{sales_channel_id}';")
        
        res_chronos = cursor.fetchall()
        
        if len(res_chronos) == 0:
            continue
        
        df_chronos = pd.DataFrame(res_chronos, columns=['start', 'sales_high', 'sales_low', 'sales_mean', 'orders_high', 'orders_low', 'orders_mean']).sort_values('start').set_index('start')
        df_chronos.index = pd.to_datetime(df_chronos.index.tz_convert(SAO_PAULO_TZ).strftime("%Y-%m-%d %H:00:00"), format='%Y-%m-%d %H:%M:%S')
        
        df_chronos = df_chronos.iloc[-24:].astype(float)
        
        if df_chronos['sales_mean'].isna().any():
            sales_na = df_chronos['sales_mean'].isna()
            df_chronos.loc[sales_na, 'sales_mean'] = 0.5*df_chronos.loc[sales_na, 'sales_high'] + 0.5*df_chronos.loc[sales_na, 'sales_low']
        if df_chronos['orders_mean'].isna().any():
            orders_na = df_chronos['orders_mean'].isna()
            df_chronos.loc[orders_na, 'orders_mean'] = 0.5*df_chronos.loc[orders_na, 'orders_high'] + 0.5*df_chronos.loc[orders_na, 'orders_low']
        
        df_client_mod.rename(columns={'n_orders': 'orders_actual', 'price_total_agg': 'sales_actual'}, inplace=True)
        forecast = df_chronos.merge(df_client_mod, how='left', left_index=True, right_index=True).fillna(0)
        
        # plt.figure(figsize=(12, 3))
        # plt.plot(forecast['sales_mean'], linestyle='--')
        # plt.plot(forecast['sales_actual'])
        # plt.fill_between(forecast.index, forecast['sales_low'], forecast['sales_high'], color='gray', alpha=0.8)
        # plt.fill_between(forecast.index, 2*forecast['sales_low']-forecast['sales_mean'], 2*forecast['sales_high']-forecast['sales_mean'], color='gray', alpha=0.5)
        # plt.fill_between(forecast.index, 3*forecast['sales_low']-2*forecast['sales_mean'], 3*forecast['sales_high']-2*forecast['sales_mean'], color='gray', alpha=0.3)
        # plt.show();
        
        # plt.figure(figsize=(12, 3))
        # plt.plot(forecast['orders_mean'], linestyle='--')
        # plt.plot(forecast['orders_actual'])
        # plt.fill_between(forecast.index, forecast['orders_low'], forecast['orders_high'], color='gray', alpha=0.8)
        # plt.fill_between(forecast.index, 2*forecast['orders_low']-forecast['orders_mean'], 2*forecast['orders_high']-forecast['orders_mean'], color='gray', alpha=0.5)
        # plt.fill_between(forecast.index, 3*forecast['orders_low']-2*forecast['orders_mean'], 3*forecast['orders_high']-2*forecast['orders_mean'], color='gray', alpha=0.3)
        # plt.show();
        
        df_forecast = forecast.assign(
            sales_covered_pts1=lambda x:
                4*x['sales_actual'].between(x['sales_low'], x['sales_high'], inclusive='both') +
                2*(x['sales_actual'].between(2*x['sales_low']-x['sales_mean'], x['sales_low'], inclusive='left') + x['sales_actual'].between(x['sales_high'], 2*x['sales_high']-x['sales_mean'], inclusive='right')) +
                1*(x['sales_actual'].between(3*x['sales_low']-2*x['sales_mean'], 2*x['sales_low']-x['sales_mean'], inclusive='left') + x['sales_actual'].between(2*x['sales_high']-x['sales_mean'], 3*x['sales_high']-2*x['sales_mean'], inclusive='right')),
            sales_covered_pts2=lambda x: x['sales_actual'].between(3*x['sales_low']-2*x['sales_mean'], 3*x['sales_high']-2*x['sales_mean'], inclusive='both'),
            sales_covered_width=lambda x: x['sales_high'] - x['sales_low'],
            orders_covered_pts1=lambda x:
                4*x['orders_actual'].between(x['orders_low'], x['orders_high'], inclusive='both') +
                2*(x['orders_actual'].between(2*x['orders_low']-x['orders_mean'], x['orders_low'], inclusive='left') + x['orders_actual'].between(x['orders_high'], 2*x['orders_high']-x['orders_mean'], inclusive='right')) +
                1*(x['orders_actual'].between(3*x['orders_low']-2*x['orders_mean'], 2*x['orders_low']-x['orders_mean'], inclusive='left') + x['orders_actual'].between(2*x['orders_high']-x['orders_mean'], 3*x['orders_high']-2*x['orders_mean'], inclusive='right')),
            orders_covered_pts2=lambda x: x['orders_actual'].between(3*x['orders_low']-2*x['orders_mean'], 3*x['orders_high']-2*x['orders_mean'], inclusive='both'),
            orders_covered_width=lambda x: x['orders_high'] - x['orders_low']
        )
                
        df_forecast_metrics = {}
        df_forecast_metrics['sales_total_covered1'] = df_forecast['sales_covered_pts1'].sum()
        df_forecast_metrics['sales_median_covered1'] = df_forecast['sales_covered_pts1'].median()
        df_forecast_metrics['sales_total_covered2'] = df_forecast['sales_covered_pts2'].sum()
        df_forecast_metrics['sales_median_covered2'] = df_forecast['sales_covered_pts2'].median()
        df_forecast_metrics['sales_median_covered_width'] = df_forecast['sales_covered_width'].median()
        df_forecast_metrics['orders_total_covered1'] = df_forecast['orders_covered_pts1'].sum()
        df_forecast_metrics['orders_median_covered1'] = df_forecast['orders_covered_pts1'].median()
        df_forecast_metrics['orders_total_covered2'] = df_forecast['orders_covered_pts2'].sum()
        df_forecast_metrics['orders_median_covered2'] = df_forecast['orders_covered_pts2'].median()
        df_forecast_metrics['orders_median_covered_width'] = df_forecast['orders_covered_width'].median()
        
        df_forecast_sales_metrics1_norm = df_forecast_metrics['sales_median_covered1'] / (1 + np.log(1+df_forecast_metrics['sales_median_covered_width']))
        df_forecast_orders_metrics1_norm = df_forecast_metrics['orders_median_covered1'] / (1 + np.log(1+df_forecast_metrics['orders_median_covered_width']))
        
        df_forecast_sales_metrics2_norm = 10*df_forecast_metrics['sales_median_covered2'] / (1 + np.log(1+df_forecast_metrics['sales_median_covered_width']))
        df_forecast_orders_metrics2_norm = 10*df_forecast_metrics['orders_median_covered2'] / (1 + np.log(1+df_forecast_metrics['orders_median_covered_width']))

        weight_chan = weights_chan_df.loc[(weights_chan_df['account_id'] == account_id) & (weights_chan_df['sales_channel_id'] == sales_channel_id), 'weights'].values[0]

        df_forecast_metrics1_norm[model][account_id][sales_channel_id] = weight_chan * (1/3*df_forecast_sales_metrics1_norm + 2/3*df_forecast_orders_metrics1_norm)
        # df_forecast_metrics1_norm[model][(account_id, sales_channel_id)] = weight_chan * (1/3*df_forecast_sales_metrics1_norm + 2/3*df_forecast_sales_metrics2_norm)
        
        df_forecast_metrics2_norm[model][account_id][sales_channel_id] = weight_chan * (1/3*df_forecast_orders_metrics2_norm + 2/3*df_forecast_sales_metrics2_norm)
        # df_forecast_metrics2_norm[model][(account_id, sales_channel_id)] = weight_chan * (1/3*df_forecast_orders_metrics2_norm + 2/3*df_forecast_sales_metrics2_norm)
        
    scores1[model] = np.sum([weights_acc_df[acc_id] * np.sum(list(df_forecast_metrics1_norm[model][acc_id].values())) for acc_id in df_forecast_metrics1_norm[model].keys()])
    scores2[model] = np.sum([weights_acc_df[acc_id] * np.sum(list(df_forecast_metrics2_norm[model][acc_id].values())) for acc_id in df_forecast_metrics2_norm[model].keys()])




amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)
f9129295-bb08-4330-b60c-9f0beadda521 42
b009c05b-326e-4ebe-988c-8d2229ec3a4c ALL
22fde42e-45d7-46cd-9d6f-3a4dbabbc579 6
f9129295-bb08-4330-b60c-9f0beadda521 24
60ca8452-5b14-481f-a637-8756f527aee8 1
457bce7b-9300-4c10-9a97-070b3c0d081d ALL
f9129295-bb08-4330-b60c-9f0beadda521 4
92f116fe-5818-41db-9eb5-87445ad3818e ALL
f9129295-bb08-4330-b60c-9f0beadda521 36
ca35e88d-d191-4722-b90c-92f4a249869b ALL
7412074f-0f3a-4724-b035-7fb4db3174c8 8
22fde42e-45d7-46cd-9d6f-3a4dbabbc579 ALL
423a069b-27b8-44ed-9ef3-ca3cf9470970 ALL
478cd985-f4fc-45b4-ac71-bf78cf11e07b 2
9d1296ca-9e4f-4840-b7b5-e12172acab78 ALL
173e2b27-8bc6-44d1-b581-a1913d6b1894 5
9d39683a-47b3-4a18-942d-f1d741a47e8c ALL
d639773a-df2c-421e-ac87-6918a754572f 1
173e2b27-8bc6-44d1-b581-a1913d6b1894 16
8fa00df1-38eb-426f-81a6-1185983e14cb ALL
3dc15e4b-b60d-4c49-aae8-97cf9664af51 ALL
6276b87c-ebb2-11ed-a05b-0242ac120003 3
789ff5c4-21d8-4ed3-ab64-8b46cf2eba9b 1
7db6eaca-fe22-41d7-b

In [53]:
scores1

{'amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)': np.float64(0.43825099558580705),
 'MEDIAN_MAD_60': np.float64(0.6451040597866221)}

In [54]:
scores2

{'amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)': np.float64(1.4728190469494846),
 'MEDIAN_MAD_60': np.float64(1.2890197171631441)}

In [55]:
joblib.dump(pd.DataFrame(scores1, index=['weights']).T, 'scores21.pkl')
joblib.dump(pd.DataFrame(scores2, index=['weights']).T, 'scores22.pkl');

In [59]:
df1 = pd.DataFrame(df_forecast_metrics1_norm)

df1

,"amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)",MEDIAN_MAD_60
f9129295-bb08-4330-b60c-9f0beadda521,"{42: 0.0, 24: 0.0, 4: 0.0, 36: 0.0, 32: 0.0, 6...","{42: 0.0, 24: 0.0, 4: 0.0, 36: 0.0, 32: 0.0, 6..."
b009c05b-326e-4ebe-988c-8d2229ec3a4c,"{'ALL': 0.6048487402587153, 1: 0.4052559294443...","{'ALL': 0.5914142111341544, 1: 0.5921084575338..."
22fde42e-45d7-46cd-9d6f-3a4dbabbc579,"{6: 0.0, 'ALL': 0.8159645645212904, 5: 0.0, 1:...","{6: 0.0, 'ALL': 1.2074720313003113, 5: 0.0, 1:..."
60ca8452-5b14-481f-a637-8756f527aee8,"{1: 0.0, 10: 0.0, 6: 0.0, 'ALL': 0.0}","{1: 0.0, 10: 0.0, 6: 0.0, 'ALL': 0.0}"
457bce7b-9300-4c10-9a97-070b3c0d081d,"{'ALL': 0.0, 5: 0.0, 1: 0.0, 3: 0.0}","{'ALL': 0.0, 5: 0.0, 1: 0.0, 3: 0.0}"
92f116fe-5818-41db-9eb5-87445ad3818e,"{'ALL': 0.3980062588658285, 5: 0.3940969897461...","{'ALL': 0.5936211452606326, 5: 0.5526117027648..."
ca35e88d-d191-4722-b90c-92f4a249869b,"{'ALL': 0.42338653946277055, 1: 0.290296649388...","{'ALL': 0.7184994600567162, 1: 0.4775179825744..."
7412074f-0f3a-4724-b035-7fb4db3174c8,"{8: 0.0, 15: 0.0, 'ALL': 0.7806872480933718, 1...","{8: 0.0, 15: 0.0, 'ALL': 0.5766440820630965, 1..."
423a069b-27b8-44ed-9ef3-ca3cf9470970,"{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}"
478cd985-f4fc-45b4-ac71-bf78cf11e07b,"{2: 0.0, 5: 0.0, 1: 0.3751711466950555, 6: 0.0...","{2: 0.0, 5: 0.0, 1: 0.29732228838711444, 6: 0...."


In [57]:
df2 = pd.DataFrame(df_forecast_metrics2_norm)

df2

,"amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)",MEDIAN_MAD_60
f9129295-bb08-4330-b60c-9f0beadda521,"{42: 0.0, 24: 0.0, 4: 0.0, 36: 0.0, 32: 0.0, 6...","{42: 0.0, 24: 0.0, 4: 0.0, 36: 0.0, 32: 0.0, 6..."
b009c05b-326e-4ebe-988c-8d2229ec3a4c,"{'ALL': 1.2907349833854043, 1: 1.2760854861992...","{'ALL': 1.040949588984497, 1: 1.0444208209831278}"
22fde42e-45d7-46cd-9d6f-3a4dbabbc579,"{6: 0.0, 'ALL': 2.8169549950638304, 5: 0.0, 1:...","{6: 0.0, 'ALL': 2.2051172231290073, 5: 0.0, 1:..."
60ca8452-5b14-481f-a637-8756f527aee8,"{1: 0.0, 10: 0.0, 6: 0.0, 'ALL': 0.0}","{1: 0.0, 10: 0.0, 6: 0.0, 'ALL': 0.0}"
457bce7b-9300-4c10-9a97-070b3c0d081d,"{'ALL': 0.0, 5: 0.0, 1: 0.0, 3: 0.0}","{'ALL': 0.0, 5: 0.0, 1: 0.0, 3: 0.0}"
92f116fe-5818-41db-9eb5-87445ad3818e,"{'ALL': 1.331617950755854, 5: 1.32016976073687...","{'ALL': 1.0519842596168885, 5: 1.0551669387140..."
ca35e88d-d191-4722-b90c-92f4a249869b,"{'ALL': 1.4703947714688186, 1: 1.0028923441229...","{'ALL': 1.2931515402600515, 1: 0.8546927395233..."
7412074f-0f3a-4724-b035-7fb4db3174c8,"{8: 0.0, 15: 0.0, 'ALL': 2.6089445802964626, 1...","{8: 0.0, 15: 0.0, 'ALL': 2.0439855290556075, 1..."
423a069b-27b8-44ed-9ef3-ca3cf9470970,"{'ALL': 0.0, 2: 0.0, 1: 0.0}","{'ALL': 0.0, 2: 0.0, 1: 0.0}"
478cd985-f4fc-45b4-ac71-bf78cf11e07b,"{2: 0.0, 5: 0.0, 1: 1.2863757285144768, 6: 0.0...","{2: 0.0, 5: 0.0, 1: 1.0571014171848696, 6: 0.0..."


In [58]:
joblib.dump(pd.DataFrame(df_forecast_metrics1_norm), 'df_scores21.pkl')
joblib.dump(pd.DataFrame(df_forecast_metrics2_norm), 'df_scores22.pkl');